# Punjabi Speech-to-Text with Gemma 4 (E4B / E2B)

Chunked ASR pipeline for long audio.  Processes 25 s segments in sequence,
prints each transcript immediately with timestamps and word count.

---
**Runtime**: T4 GPU  |  **License**: Apache 2.0 (accept on Hugging Face first)

### Setup
1. Accept the model license at https://huggingface.co/google/gemma-4-E4B-it
2. Create an HF token at https://huggingface.co/settings/tokens
3. In Colab: 🔑 **Secrets** → add `HF_TOKEN`
4. Upload your Punjabi `.wav` / `.mp3` file to the Colab runtime

In [ ]:
# @title 1. Install Dependencies
import subprocess, sys, importlib.metadata, warnings
warnings.filterwarnings("ignore")

REQUIRED = {
    "transformers": "5.5.0",
    "accelerate": None,
    "torch": None,
    "librosa": None,
    "soundfile": None,
}


def _ensure_deps():
    for pkg, min_ver in REQUIRED.items():
        need = False
        try:
            ver = importlib.metadata.version(pkg)
            if min_ver:
                p = tuple(int(x) for x in ver.split("."))
                r = tuple(int(x) for x in min_ver.split("."))
                if p < r:
                    need = True
        except importlib.metadata.PackageNotFoundError:
            need = True
        if need:
            spec = f"{pkg}>={min_ver}" if min_ver else pkg
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--upgrade", spec]
            )

_ensure_deps()
print("All dependencies satisfied.")

In [ ]:
# @title 2. Imports & Model Configuration
import torch
import librosa
import numpy as np
import gc
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
)

# ── Single variable to swap model ──
MODEL_ID = "google/gemma-4-E4B-it"  # or "google/gemma-4-E2B-it"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")


In [ ]:
# @title 3. Load Token, Processor & Model
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, ValueError):
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Set it in Colab Secrets (key: HF_TOKEN) "
        "or as an environment variable."
    )

print("Loading processor ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

print(f"Loading model {MODEL_ID} ...")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()
print("Model loaded successfully.")


In [ ]:
# @title 4. Audio Segmentation & Preprocessing
CHUNK_SEC = 25  # stay well under Gemma 4 30 s limit


def segment_and_prepare_audio(
    file_path: str, chunk_length_s: int = CHUNK_SEC
) -> list[dict]:
    """Load full audio, preprocess for Gemma 4, slice into chunks.

    Returns a list of dicts, each with:
      - "array":  1-D float32 numpy array (mono, 16 kHz, [-1, 1])
      - "start":  start time in seconds
      - "end":    end time in seconds
    """
    if not file_path:
        raise ValueError("file_path must be a non-empty string.")

    # librosa.load handles: resample to sr, mono downmix, float32 norm
    audio, sr = librosa.load(file_path, sr=16000, mono=True)

    total_samples = audio.shape[0]
    chunk_samples = chunk_length_s * sr

    segments = []
    for start_sample in range(0, total_samples, chunk_samples):
        end_sample = min(start_sample + chunk_samples, total_samples)
        chunk = audio[start_sample:end_sample]

        # timestamps derived from sample indices / sampling rate
        start_sec = start_sample / sr
        end_sec = end_sample / sr

        segments.append({
            "array": chunk,
            "start": start_sec,
            "end": end_sec,
        })

    return segments


In [ ]:
# @title 5. Chunked Transcription Pipeline

ASR_PROMPT = (
    "Transcribe the following speech segment in Punjabi into "
    "Gurmukhi text. Output only the raw transcript text with no "
    "introductory text, greetings, or semantic corrections."
)


def _fmt_ts(sec: float) -> str:
    """Format seconds as MM:SS."""
    m, s = divmod(int(sec), 60)
    return f"{m:02d}:{s:02d}"


def transcribe_chunked(file_path: str, max_new_tokens: int = 256) -> None:
    """Segment audio, transcribe each chunk, print results immediately."""
    segments = segment_and_prepare_audio(file_path)
    total = len(segments)
    print(f"Processing {total} chunk(s) ...\n")

    for idx, seg in enumerate(segments, start=1):
        chunk_arr = seg["array"]
        ts_start = _fmt_ts(seg["start"])
        ts_end = _fmt_ts(seg["end"])

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": ASR_PROMPT},
                    {"type": "audio", "audio": chunk_arr},
                ],
            },
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        input_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        response_ids = generated_ids[0][input_len:]
        transcript = processor.decode(
            response_ids, skip_special_tokens=True
        ).strip()

        # Word count over whitespace
        word_count = len(transcript.split()) if transcript else 0

        # ── Immediate print ──
        print("---")
        print(f"### Chunk {idx} | Timestamps: {ts_start} - {ts_end}")
        print("**Gurmukhi Transcript:**")
        print(transcript if transcript else "[empty]")
        print()
        print("**Metrics:**")
        print(f"* Word Count: {word_count}")
        print("---\n")

        # ── Memory cleanup ──
        del inputs, generated_ids, chunk_arr, messages
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# @title 6. Run Transcription
# Upload your file to Colab first, then set the path below.
AUDIO_PATH = "/content/sample_punjabi.wav"  # <-- CHANGE ME

try:
    transcribe_chunked(AUDIO_PATH)
except Exception as exc:
    print(f"ERROR: {exc}")